# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [2]:
# Quiet ensures your UI is not full of logs.
%pip install -q -U google-genai
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.
Done.


In [3]:
import os

if os.environ.get("GOOGLE_API_KEY"):
    print("API key is available!")
else:
    print("API key is missing!")

API key is available!


In [4]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "food_safety": (
        "Chicken should be cooked thoroughly before eating. "
        "The safe internal temperature for chicken is 74 degrees Celsius. "
        "A food thermometer can be used to check the temperature. "
        "Raw chicken should be kept separate from other foods."
    ),

    "pasta": (
        "Pasta should be cooked in a pot of boiling water. "
        "Add some salt to the boiling water for flavour. "
        "Stir the pasta occasionally to prevent it from sticking. "
        "Taste the pasta to check whether it is cooked properly."
    ),

    "bread": (
        "Bread is commonly made using flour, water, yeast and salt. "
        "Yeast helps the dough rise by producing carbon dioxide. "
        "The dough should be allowed to rest in a warm place. "
        "Bread is baked in an oven until its crust becomes golden brown."
    ),

    "rice": (
        "Rice should be washed before cooking to remove excess starch. "
        "The amount of water required depends on the type of rice. "
        "Rice can be cooked using a pot or a rice cooker. "
        "After cooking, allow the rice to rest before serving."
    )
}

In [2]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("pasta"))


[Source: pasta]
Pasta should be cooked in a pot of boiling water. Add some salt to the boiling water for flavour. Stir the pasta occasionally to prevent it from sticking. Taste the pasta to check whether it is cooked properly.


In [10]:
from google import genai

print("Gemini package installed successfully!")

Gemini package installed successfully!


In [8]:
# @title Ask gemini to search for something

import os
from google import genai

MODEL_NAME = "gemini-3.8-flash"

SYSTEM_INSTRUCTION = """
You are a chef training at a Michelin star restaurant.

When you need information about temperatures,
ingredients, cooking time etc., use the search_documents tool.

Only answer based on what you find in the documents.

If the documents do not contain the answer,
say so clearly.
"""

client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"]
)

print("Model ready:", MODEL_NAME)

Model ready: gemini-3.8-flash


In [14]:
import time
from google.genai import types


# Define the search tool
search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the documents for relevant information.",
            parameters_json_schema={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    }
                },
                "required": ["query"]
            }
        )
    ]
)


# Configure the model
config = types.GenerateContentConfig(
    tools=[search_tool],
    system_instruction=SYSTEM_INSTRUCTION
)


# Handle API retries
def send_with_retry(contents, max_retries=3):

    for attempt in range(max_retries):

        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )

        except Exception as e:

            if "429" in str(e) and attempt < max_retries - 1:

                wait = 20 * (attempt + 1)

                print("Rate limit hit. Waiting", wait, "seconds...")

                time.sleep(wait)

            else:
                raise


# Main ReAct agent
def run_agent(question, max_steps=8):

    print("Question:", question)
    print("-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=question)
            ]
        )
    ]

    for step in range(1, max_steps + 1):

        response = send_with_retry(contents)

        model_content = response.candidates[0].content

        contents.append(model_content)

        calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call
        ]

        # Check whether the model has finished answering
        if not calls:

            print("\nFinal answer:")
            print(response.text)

            return response.text

        result_parts = []

        # Execute the requested search
        for call in calls:

            args = dict(call.args)

            print("Step", step, ":", call.name, args)

            result = search_documents(**args)

            print("Search result:", result[:150])

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        # Send search results back to Gemini
        contents.append(
            types.Content(
                role="tool",
                parts=result_parts
            )
        )

    return "Stopped: maximum steps reached."


# Test the agent
_ = run_agent(
    "What is the safe internal temperature for chicken?"
)

Question: What is the safe internal temperature for chicken?
--------------------------------------------------

[ERROR] Permission Denied (403). Your API Key doesn't have access to the Gemini free tier in this project.
To fix this: Go to https://aistudio.google.com/, create a NEW API key (select 'Create API key in new project'), and re-run your setup cell with the new key.


In [17]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
# Ask follow up questions

questions = [
    "What is the safe internal temperature for chicken?",
    "How can I prevent pasta from sticking while cooking?",
    "What does yeast do in bread dough?",
    "What should I do before cooking rice?"
]

for i, q in enumerate(questions):

    if i > 0:
        time.sleep(15)

    print("\n" + "=" * 60)

    run_agent(q)


Question: What is the safe internal temperature for chicken?
--------------------------------------------------

[ERROR] Permission Denied (403) while asking: 'What is the safe internal temperature for chicken?'
Your API Key doesn't have access to the Gemini free tier in this project.
Please create a NEW API key (select 'Create API key in new project') on https://aistudio.google.com/ and update your setup.


In [21]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)